# Detecting Cascading Failures in Distributed Systems

In [ ]:
# 1. Define microservice graph manually
# We define 8 nodes from scratch without relying on a preset.
from causalnerve import CausalNerve
import numpy as np

services = ['API Gateway', 'Auth', 'Database', 'Cache', 'Message Queue', 'Worker', 'Monitoring', 'Load Balancer']
nerve = CausalNerve(nodes=8)
nerve.node_labels = {i: name for i, name in enumerate(services)}

In [ ]:
# 2. Simulate normal operations
# Generate 100 cycles of healthy traffic metrics.
normal_metrics = [np.random.normal(0, 1, 8) for _ in range(100)]
nerve.fit(normal_metrics, epochs=10)

In [ ]:
# 3. Inject database failure
# At cycle 50, simulate a latency spike in the database (node 2) that cascades.
faulty_metrics = []
for i in range(100):
    base = np.random.normal(0, 1, 8)
    if i >= 50:
        base[2] += 5.0 # DB latency spikes
        base[0] += 3.0 # API gateway waits on DB
        base[4] += 2.0 # Queue backs up
    faulty_metrics.append(base)

nerve.watch(faulty_metrics)

In [ ]:
# 4. Identify first alarming service
# Which service triggered the structural health failure first?
health = nerve.structural_health()
print('Alarms triggered on:', [nerve.node_labels[i] for i in health.alarms_active])

In [ ]:
# 5. Root Cause isolation
# Trace back from the visible API Gateway (node 0) alarm to find the root.
print('\nTracing root cause from API Gateway:')
nerve.why(0)

In [ ]:
# 6. What-If Simulation
# What happens if we intervene and fix the database latency?
print('\nSimulating intervention on Database (latency = 0.0):')
nerve.what_if(2, 0.0)

In [ ]:
# 7. Compare: Root cause vs Symptoms
# Note how traditional monitoring alerts on API Gateway, but CausalNerve targets the DB.
print('Root Cause identified: Database. Visible symptoms observed: API Gateway, Queue.')

### Why this matters for SRE teams
In a microservice environment, a single bottleneck causes a flood of secondary alerts (symptoms). By maintaining a real-time structural map, CausalNerve instantly isolates the precise root service, turning hours of log-digging into an automated, targeted response.